# Network Expansion
## Model 3 (Test) - Stochastic Intertemporal Model

Scenario-based, multi-year formulation: investments (substations, lines, reinforcements) are shared across scenarios and decided once; operations are scenario-specific. Costs are discounted and budgets are enforced per year. Uncertainty enters via demand scenarios with probabilities.

### 1 - Imports

In [36]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
from src.solver import solve_network_stochastic

### 2 - Define the Distribution Network (shared)

In [37]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial substation
S1 = Substation("S1", "N4", 40, ['N3', 'N5', 'N9'], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50

base_load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5','N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    base_load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations
capacity = 15
s_cost = 100       # activation cost
l_cost = line_cost # feeder line cost
r_cost = 200       # capacity reinforcement cost

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 3 - Scenarios and horizon

In [38]:
years = list(range(1, 11))      # Years 1..10
R = 10                             # Capacity reinforcement size
dr = 0.05                          # Discount rate
# Dynamic budgets per year
B = {t: 180 + 25*(t-1) for t in years}

# Scenario growth rates (annual) and probabilities
scenarios_def = {
    'conservative': {'prob': 0.25, 'growth': 0.02},
    'base':         {'prob': 0.50, 'growth': 0.06},
    'high':         {'prob': 0.25, 'growth': 0.089},
}

# Build per-scenario, per-year demands
scenarios = {}
for name, data in scenarios_def.items():
    g = data['growth']
    scenarios[name] = {'prob': data['prob'], 'demands': {}}
    for t in years:
        factor = (1 + g) ** (t - 1)
        scenarios[name]['demands'][t] = {ld: val * factor for ld, val in base_load_capacity.items()}

# Quick demand check
total_demand = {name: {t: round(sum(scenarios[name]['demands'][t].values()), 2) for t in years} for name in scenarios}
total_demand

# Connection change penalties
connect_cost = 25
disconnect_cost = 0

### 4 - Solve stochastic model

In [39]:
solution = solve_network_stochastic(
    Network=DistributionNetwork,
    R=R,
    B=B,
    dr=dr,
    years=years,
    scenarios=scenarios,
    op_cost=10,
    reliability_target= 1,
    shed_cost=0,
    connect_cost=connect_cost,
    disconnect_cost=disconnect_cost,
    OutputFlag=0
)


### 5 - Investment summary (scenario-invariant)

In [40]:
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
base_edge_cost = DistributionNetwork.edge_cost

# New lines built per year (costed edges only)
lines_added = {}
for t in years:
    added = set()
    for (i, j, s, tau), val in solution['b_on'].items():
        if tau == t and val > 0.5:
            e = (i, j)
            if base_edge_cost.get(e, 0) > 0:
                added.add(e)
    lines_added[t] = sorted(list(added))

# Expected demand and supply across scenarios
expected_demand = {}
expected_supply = {}
for t in years:
    exp_d = 0.0
    exp_s = 0.0
    for omega, data in scenarios.items():
        p = data['prob']
        exp_d += p * sum(data['demands'][t].values())
        exp_s += p * sum(solution['r'][(s, t, omega)] for s in S_idx)
    expected_demand[t] = round(exp_d, 2)
    expected_supply[t] = round(exp_s, 2)

investment_summary = pd.DataFrame({
    'Budget': [B[t] if not isinstance(B, dict) else B[t] for t in years],
    'System Demand (exp)': [expected_demand[t] for t in years],
    'System Supply (exp)': [expected_supply[t] for t in years],
    'System Capacity': [sum(solution['P'][(s, t)] for s in S_idx) for t in years],
    'Substations Active': [[s for s in S_idx if solution['w'][(s, t)] > 0.5] for t in years],
    'Lines Added': [lines_added[t] for t in years],
    'Cost substation (nom)': [round(solution['cost_components_nominal'][t]['substation'], 2) for t in years],
    'Cost lines (nom)': [round(solution['cost_components_nominal'][t]['lines'], 2) for t in years],
    'Cost connect (nom)': [round(solution['cost_components_nominal'][t]['connect'], 2) for t in years],
    'Cost disconnect (nom)': [round(solution['cost_components_nominal'][t]['disconnect'], 2) for t in years],
    'Cost reinforcement (nom)': [round(solution['cost_components_nominal'][t]['reinforcement'], 2) for t in years],
    'Cost opex (nom)': [round(solution['cost_components_nominal'][t]['opex'], 2) for t in years],
    'Cost (nominal)': [round(solution['cost_per_year_nominal'][t], 2) for t in years],
    'Cost (discounted)': [round(solution['cost_per_year'][t], 2) for t in years]
}, index=years)

investment_totals = pd.DataFrame({
    'Budget': [''],
    'System Demand (exp)': [''],
    'System Supply (exp)': [''],
    'System Capacity': [''],
    'Substations Active': [''],
    'Lines Added': [''],
    'Cost substation (nom)': [round(sum(solution['cost_components_nominal'][t]['substation'] for t in years), 2)],
    'Cost lines (nom)': [round(sum(solution['cost_components_nominal'][t]['lines'] for t in years), 2)],
    'Cost connect (nom)': [round(sum(solution['cost_components_nominal'][t]['connect'] for t in years), 2)],
    'Cost disconnect (nom)': [round(sum(solution['cost_components_nominal'][t]['disconnect'] for t in years), 2)],
    'Cost reinforcement (nom)': [round(sum(solution['cost_components_nominal'][t]['reinforcement'] for t in years), 2)],
    'Cost opex (nom)': [round(sum(solution['cost_components_nominal'][t]['opex'] for t in years), 2)],
    'Cost (nominal)': [round(sum(solution['cost_per_year_nominal'][t] for t in years), 2)],
    'Cost (discounted)': [round(sum(solution['cost_per_year'][t] for t in years), 2)]
}, index=['Total'])

investment_summary = pd.concat([investment_summary, investment_totals])
investment_summary


,Budget,System Demand (exp),System Supply (exp),System Capacity,Substations Active,Lines Added,Cost substation (nom),Cost lines (nom),Cost connect (nom),Cost disconnect (nom),Cost reinforcement (nom),Cost opex (nom),Cost (nominal),Cost (discounted)
1,180,39.0,39.0,55.0,"[1, 2]","[(2, 14)]",100.0,50.0,0.00,0.0,0.0,20.0,170.00,170.00
2,205,41.23,41.23,55.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,19.05
3,230,43.62,43.62,55.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,18.14
4,255,46.16,46.16,55.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,17.28
5,280,48.88,48.88,65.0,"[1, 2]",[],0.0,0.0,0.00,0.0,200.0,20.0,220.00,180.99
6,305,51.79,51.79,65.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,15.67
7,330,54.9,54.9,75.0,"[1, 2]",[],0.0,0.0,6.25,0.0,200.0,20.0,226.25,168.83
8,355,58.23,58.23,90.0,"[1, 2, 4]","[(13, 16)]",100.0,50.0,25.00,0.0,0.0,30.0,205.00,145.69
9,380,61.79,61.79,90.0,"[1, 2, 4]",[],0.0,0.0,0.00,0.0,0.0,30.0,30.00,20.31
10,405,65.6,65.6,90.0,"[1, 2, 4]",[],0.0,0.0,0.00,0.0,0.0,30.0,30.00,19.34


### 6 - Scenario operations

In [41]:
records = []
for omega, data in scenarios.items():
    for t in years:
        demand = sum(data['demands'][t].values())
        supply = sum(solution['r'][(s, t, omega)] for s in S_idx)
        shed = sum(solution['ls'][(n, t, omega)] for n in range(1, len(DistributionNetwork.NODES)+1))
        reliability = 1 - shed / demand if demand > 0 else 1.0
        active = [s for s in S_idx if solution['w'][(s, t)] > 0.5]
        lines_added = [ (i,j) for (i,j,s,tau), val in solution['b_on'].items() if tau == t and val > 0.5 and base_edge_cost.get((i,j),0) > 0 ]
        records.append({
            'Scenario': omega,
            'Prob': data['prob'],
            'Year': t,
            'Budget': B[t] if not isinstance(B, dict) else B[t],
            'Demand': round(demand, 2),
            'Supply': round(supply, 2),
            'System Capacity': sum(solution['P'][(s, t)] for s in S_idx),
            'Substations Active': active,
            'Lines Added': sorted(list(set(lines_added))),
            'Cost substation (nom)': round(solution['cost_components_nominal'][t]['substation'], 2),
            'Cost lines (nom)': round(solution['cost_components_nominal'][t]['lines'], 2),
            'Cost connect (nom)': round(solution['cost_components_nominal'][t]['connect'], 2),
            'Cost disconnect (nom)': round(solution['cost_components_nominal'][t]['disconnect'], 2),
            'Cost reinforcement (nom)': round(solution['cost_components_nominal'][t]['reinforcement'], 2),
            'Cost opex (nom)': round(solution['cost_components_nominal'][t]['opex'], 2),
            'Cost (nominal)': round(solution['cost_per_year_nominal'][t], 2),
            'Cost (discounted)': round(solution['cost_per_year'][t], 2)
        })

ops_summary = pd.DataFrame(records)
ops_summary


,Scenario,Prob,Year,Budget,Demand,Supply,System Capacity,Substations Active,Lines Added,Cost substation (nom),Cost lines (nom),Cost connect (nom),Cost disconnect (nom),Cost reinforcement (nom),Cost opex (nom),Cost (nominal),Cost (discounted)
0,conservative,0.25,1,180,39.00,39.00,55.0,"[1, 2]","[(2, 14)]",100.0,50.0,0.00,0.0,0.0,20.0,170.00,170.00
1,conservative,0.25,2,205,39.78,39.78,55.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,19.05
2,conservative,0.25,3,230,40.58,40.58,55.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,18.14
3,conservative,0.25,4,255,41.39,41.39,55.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,17.28
4,conservative,0.25,5,280,42.21,42.21,65.0,"[1, 2]",[],0.0,0.0,0.00,0.0,200.0,20.0,220.00,180.99
5,conservative,0.25,6,305,43.06,43.06,65.0,"[1, 2]",[],0.0,0.0,0.00,0.0,0.0,20.0,20.00,15.67
6,conservative,0.25,7,330,43.92,43.92,75.0,"[1, 2]",[],0.0,0.0,6.25,0.0,200.0,20.0,226.25,168.83
7,conservative,0.25,8,355,44.80,44.80,90.0,"[1, 2, 4]","[(13, 16)]",100.0,50.0,25.00,0.0,0.0,30.0,205.00,145.69
8,conservative,0.25,9,380,45.69,45.69,90.0,"[1, 2, 4]",[],0.0,0.0,0.00,0.0,0.0,30.0,30.00,20.31
9,conservative,0.25,10,405,46.61,46.61,90.0,"[1, 2, 4]",[],0.0,0.0,0.00,0.0,0.0,30.0,30.00,19.34


### 7 - Final-year assignments per scenario

In [42]:
last_year = years[-1]
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
N_idx = list(range(1, len(DistributionNetwork.NODES)+1))
for omega in scenarios:
    print(f"Assignments in Year {last_year} for scenario {omega}:")
    for n in N_idx:
        for s in S_idx:
            if solution['y'][(n, s, last_year, omega)] > 0.5:
                print(f"Node {DistributionNetwork.NODES[n-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Assignments in Year 10 for scenario conservative:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S4
Node N14 assigned to S2
Node N16 assigned to S4
Assignments in Year 10 for scenario base:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S4
Node N14 assigned to S2
Node N16 assigned to S4
Assignments in Year 10 for scenario high:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S2
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Nod